# AB Test Core
*Cookbook: `06a_statistical_testing/`*

Randomization check → power analysis → hypothesis test → effect size.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import matplotlib.pyplot as plt, seaborn as sns

# ── Randomization Validation (check covariate balance) ────────────────────
def randomization_check(df, group_col, covariates):
    control   = df[df[group_col] == 'control']
    treatment = df[df[group_col] == 'treatment']
    results = []
    for col in covariates:
        t, p = stats.ttest_ind(control[col].dropna(), treatment[col].dropna())
        results.append({'covariate': col, 't_stat': round(t,4), 'p_value': round(p,4),
                        'balanced': '✓' if p > 0.05 else '✗ IMBALANCED'})
    return pd.DataFrame(results)

# display(randomization_check(df, 'group', ['age','income','tenure']))


In [ ]:
# ── Power Analysis ────────────────────────────────────────────────────────
from statsmodels.stats.power import TTestIndPower

def power_analysis(effect_size=0.2, alpha=0.05, power=0.80):
    analysis = TTestIndPower()
    n = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power)
    print(f'Effect size: {effect_size}  |  Alpha: {alpha}  |  Power: {power}')
    print(f'Required sample per group: {np.ceil(n):.0f}')
    return n

# power_analysis(effect_size=0.2)  # small effect
# power_analysis(effect_size=0.5)  # medium effect


In [ ]:
# ── Two-Sample T-Test ─────────────────────────────────────────────────────
def run_ab_test(control, treatment, metric, alpha=0.05):
    c = control[metric].dropna()
    t = treatment[metric].dropna()
    t_stat, p_val = stats.ttest_ind(c, t)
    ate = t.mean() - c.mean()
    ci = stats.t.interval(1-alpha, df=len(c)+len(t)-2,
                           loc=ate, scale=stats.sem(np.concatenate([c,t])))
    print(f'Control mean:   {c.mean():.4f}')
    print(f'Treatment mean: {t.mean():.4f}')
    print(f'ATE:  {ate:.4f}')
    print(f'95% CI: ({ci[0]:.4f}, {ci[1]:.4f})')
    print(f't={t_stat:.4f}, p={p_val:.4f}')
    print('✓ Significant' if p_val < alpha else '✗ Not significant')

# control   = df[df['group']=='control']
# treatment = df[df['group']=='treatment']
# run_ab_test(control, treatment, metric='conversion')


In [ ]:
# ── Chi-Square Test (binary / proportion metrics) ─────────────────────────
# from scipy.stats import chi2_contingency
# contingency = pd.crosstab(df['group'], df['converted'])
# chi2, p, dof, expected = chi2_contingency(contingency)
# print(f'chi2={chi2:.4f}, p={p:.4f}')
